# Gradient Boosting Regression

A comprehensive guide to understanding and implementing Gradient Boosting for regression tasks.

## Table of Contents

1. [Theory Section](#1.-Theory-Section)
2. [Implementation from Scratch](#2.-Implementation-from-Scratch)
3. [Training & Optimization](#3.-Training-&-Optimization)
4. [Diagnostics & Evaluation](#4.-Diagnostics-&-Evaluation)
5. [Visualizations](#5.-Visualizations)
6. [Use Cases & Guidelines](#6.-Use-Cases-&-Guidelines)
7. [Comparison with sklearn](#7.-Comparison-with-sklearn)

---

## 1. Theory Section

### 1.1 The Boosting Concept

**Boosting** is an ensemble learning technique that combines multiple weak learners sequentially to create a strong learner. Unlike bagging (e.g., Random Forest) where models are trained independently, boosting trains models **iteratively**, with each new model focusing on correcting the errors of the previous ensemble.

Key idea: **"Wisdom of crowds through sequential improvement"**

The ensemble prediction is the sum of all weak learner predictions:

$$F(x) = \sum_{m=1}^{M} f_m(x)$$

where $f_m(x)$ is the $m$-th weak learner.

### 1.2 Gradient Descent in Function Space

Gradient Boosting performs gradient descent in **function space** rather than parameter space. At each iteration:

1. Compute the **negative gradient** (pseudo-residuals) of the loss function with respect to the current predictions
2. Fit a new weak learner to these pseudo-residuals
3. Add the new learner to the ensemble with a learning rate

**Mathematical formulation:**

For a loss function $L(y, F(x))$, the pseudo-residuals are:

$$r_{im} = -\left[\frac{\partial L(y_i, F(x_i))}{\partial F(x_i)}\right]_{F=F_{m-1}}$$

For **MSE loss** $L = \frac{1}{2}(y - F(x))^2$:

$$r_{im} = y_i - F_{m-1}(x_i)$$

This is simply the **residual** - the difference between actual and predicted values!

### 1.3 Weak Learners

Gradient Boosting typically uses **decision trees** as weak learners, usually:

- **Decision stumps**: Trees with depth 1 (single split)
- **Shallow trees**: Trees with max_depth 3-6

Why weak learners?
- Individually they have high bias but low variance
- The boosting process reduces bias while maintaining low variance
- Deep trees would overfit to residuals too quickly

### 1.4 Learning Rate and Shrinkage

The **learning rate** (also called shrinkage, $\eta$) controls how much each weak learner contributes:

$$F_m(x) = F_{m-1}(x) + \eta \cdot f_m(x)$$

- **Small learning rate** (0.01-0.1): Slower convergence, better generalization, needs more trees
- **Large learning rate** (0.5-1.0): Faster convergence, higher risk of overfitting

**Trade-off**: Lower learning rate + more estimators = better performance but longer training

### 1.5 Loss Functions

Gradient Boosting is flexible with loss functions. Common choices for regression:

| Loss Function | Formula | Use Case |
|--------------|---------|----------|
| **MSE (ls)** | $\frac{1}{2}(y-F)^2$ | General regression |
| **MAE (lad)** | $|y-F|$ | Robust to outliers |
| **Huber** | MSE if small, MAE if large | Balance of both |
| **Quantile** | Asymmetric piecewise linear | Prediction intervals |

The gradient of MSE is simply the residual, making it computationally efficient.

### 1.6 Gradient Boosting Algorithm

```
Algorithm: Gradient Boosting for Regression
-------------------------------------------
Input: Training data {(x_i, y_i)}, loss function L, number of iterations M, learning rate eta

1. Initialize: F_0(x) = mean(y)  # or argmin_gamma sum(L(y_i, gamma))

2. For m = 1 to M:
   a. Compute pseudo-residuals:
      r_im = -[dL(y_i, F(x_i))/dF(x_i)] at F=F_{m-1}
   
   b. Fit weak learner f_m to pseudo-residuals {(x_i, r_im)}
   
   c. Update model:
      F_m(x) = F_{m-1}(x) + eta * f_m(x)

3. Output: F_M(x)
```

---

## 2. Implementation from Scratch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

### 2.1 Decision Tree Regressor (Weak Learner)

In [ ]:
class DecisionTreeRegressor:
    """
    A simple Decision Tree Regressor for use as a weak learner in Gradient Boosting.
    Uses recursive binary splitting with MSE as the splitting criterion.
    """
    
    def __init__(self, max_depth=3, min_samples_split=2):
        """
        Initialize the decision tree.
        
        Parameters:
        -----------
        max_depth : int
            Maximum depth of the tree. Shallow trees (1-3) work best for boosting.
        min_samples_split : int
            Minimum samples required to split a node.
        """
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None
        
    def _mse(self, y):
        """Calculate MSE of predictions (variance of y)."""
        if len(y) == 0:
            return 0
        return np.var(y)
    
    def _find_best_split(self, X, y):
        """
        Find the best feature and threshold to split on.
        
        Returns:
        --------
        best_feature, best_threshold, best_gain
        """
        n_samples, n_features = X.shape
        best_gain = -np.inf
        best_feature = None
        best_threshold = None
        
        # Current node's MSE
        current_mse = self._mse(y)
        
        for feature in range(n_features):
            # Get unique thresholds (midpoints between sorted unique values)
            thresholds = np.unique(X[:, feature])
            if len(thresholds) > 10:  # Subsample thresholds for efficiency
                thresholds = np.percentile(X[:, feature], np.linspace(0, 100, 11))
            
            for threshold in thresholds:
                # Split data
                left_mask = X[:, feature] <= threshold
                right_mask = ~left_mask
                
                if np.sum(left_mask) < self.min_samples_split or np.sum(right_mask) < self.min_samples_split:
                    continue
                
                # Calculate weighted MSE after split
                n_left, n_right = np.sum(left_mask), np.sum(right_mask)
                weighted_mse = (n_left * self._mse(y[left_mask]) + 
                               n_right * self._mse(y[right_mask])) / n_samples
                
                # Information gain
                gain = current_mse - weighted_mse
                
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold
        
        return best_feature, best_threshold, best_gain
    
    def _build_tree(self, X, y, depth=0):
        """
        Recursively build the decision tree.
        
        Returns:
        --------
        dict: Tree node containing either a leaf value or split information
        """
        n_samples = len(y)
        
        # Stopping conditions: max depth reached or not enough samples
        if depth >= self.max_depth or n_samples < self.min_samples_split:
            return {'leaf': True, 'value': np.mean(y)}
        
        # Find best split
        feature, threshold, gain = self._find_best_split(X, y)
        
        # No valid split found
        if feature is None or gain <= 0:
            return {'leaf': True, 'value': np.mean(y)}
        
        # Split data
        left_mask = X[:, feature] <= threshold
        right_mask = ~left_mask
        
        # Recursively build subtrees
        left_subtree = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_subtree = self._build_tree(X[right_mask], y[right_mask], depth + 1)
        
        return {
            'leaf': False,
            'feature': feature,
            'threshold': threshold,
            'left': left_subtree,
            'right': right_subtree
        }
    
    def fit(self, X, y):
        """Fit the decision tree to training data."""
        self.tree = self._build_tree(X, y)
        return self
    
    def _predict_single(self, x, node):
        """Predict for a single sample by traversing the tree."""
        if node['leaf']:
            return node['value']
        
        if x[node['feature']] <= node['threshold']:
            return self._predict_single(x, node['left'])
        else:
            return self._predict_single(x, node['right'])
    
    def predict(self, X):
        """Predict for multiple samples."""
        return np.array([self._predict_single(x, self.tree) for x in X])

### 2.2 Gradient Boosting Regressor

In [ ]:
class GradientBoostingRegressor:
    """
    Gradient Boosting Regressor implementation from scratch.
    
    Uses MSE loss function with decision trees as weak learners.
    Implements shrinkage (learning rate) to prevent overfitting.
    """
    
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3, 
                 min_samples_split=2, verbose=False):
        """
        Initialize the Gradient Boosting Regressor.
        
        Parameters:
        -----------
        n_estimators : int
            Number of boosting iterations (weak learners).
        learning_rate : float
            Shrinkage factor. Lower values require more estimators but often
            lead to better generalization.
        max_depth : int
            Maximum depth of individual trees.
        min_samples_split : int
            Minimum samples required to split a node.
        verbose : bool
            Print training progress.
        """
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.verbose = verbose
        
        # Will store fitted trees and training history
        self.trees = []
        self.initial_prediction = None
        self.train_losses = []
        self.val_losses = []
        
    def _compute_pseudo_residuals(self, y, y_pred):
        """
        Compute pseudo-residuals (negative gradient of MSE loss).
        For MSE: gradient = -(y - y_pred), so negative gradient = y - y_pred
        """
        return y - y_pred
    
    def _mse_loss(self, y, y_pred):
        """Calculate MSE loss."""
        return np.mean((y - y_pred) ** 2)
    
    def fit(self, X, y, X_val=None, y_val=None):
        """
        Fit the gradient boosting model.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
            Training features.
        y : array-like of shape (n_samples,)
            Training targets.
        X_val : array-like, optional
            Validation features for tracking validation loss.
        y_val : array-like, optional
            Validation targets.
        
        Returns:
        --------
        self
        """
        # Reset state
        self.trees = []
        self.train_losses = []
        self.val_losses = []
        
        # Step 1: Initialize with mean of y
        self.initial_prediction = np.mean(y)
        y_pred = np.full(len(y), self.initial_prediction)
        
        # Track validation predictions if provided
        if X_val is not None:
            y_val_pred = np.full(len(y_val), self.initial_prediction)
        
        # Step 2: Iteratively add weak learners
        for m in range(self.n_estimators):
            # Compute pseudo-residuals
            residuals = self._compute_pseudo_residuals(y, y_pred)
            
            # Fit a tree to the residuals
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split
            )
            tree.fit(X, residuals)
            self.trees.append(tree)
            
            # Update predictions with shrinkage
            y_pred += self.learning_rate * tree.predict(X)
            
            # Track training loss
            train_loss = self._mse_loss(y, y_pred)
            self.train_losses.append(train_loss)
            
            # Track validation loss if provided
            if X_val is not None:
                y_val_pred += self.learning_rate * tree.predict(X_val)
                val_loss = self._mse_loss(y_val, y_val_pred)
                self.val_losses.append(val_loss)
            
            # Print progress
            if self.verbose and (m + 1) % 10 == 0:
                msg = f"Iteration {m + 1}/{self.n_estimators} - Train MSE: {train_loss:.4f}"
                if X_val is not None:
                    msg += f" - Val MSE: {val_loss:.4f}"
                print(msg)
        
        return self
    
    def predict(self, X):
        """
        Make predictions for input samples.
        
        Parameters:
        -----------
        X : array-like of shape (n_samples, n_features)
        
        Returns:
        --------
        y_pred : array of shape (n_samples,)
        """
        # Start with initial prediction
        y_pred = np.full(X.shape[0], self.initial_prediction)
        
        # Add contributions from each tree
        for tree in self.trees:
            y_pred += self.learning_rate * tree.predict(X)
        
        return y_pred
    
    def staged_predict(self, X):
        """
        Generate predictions at each stage (after each tree is added).
        Useful for analyzing the boosting process.
        
        Yields:
        -------
        y_pred : array of shape (n_samples,)
        """
        y_pred = np.full(X.shape[0], self.initial_prediction)
        
        for tree in self.trees:
            y_pred = y_pred + self.learning_rate * tree.predict(X)
            yield y_pred.copy()
    
    def feature_importances(self, X, y):
        """
        Compute feature importances using permutation importance.
        
        Parameters:
        -----------
        X : array-like
            Features to evaluate importance on.
        y : array-like
            True target values.
        
        Returns:
        --------
        importances : array of shape (n_features,)
        """
        baseline_mse = self._mse_loss(y, self.predict(X))
        importances = np.zeros(X.shape[1])
        
        for feature in range(X.shape[1]):
            # Permute feature values
            X_permuted = X.copy()
            X_permuted[:, feature] = np.random.permutation(X_permuted[:, feature])
            
            # Calculate increase in MSE
            permuted_mse = self._mse_loss(y, self.predict(X_permuted))
            importances[feature] = permuted_mse - baseline_mse
        
        # Normalize to sum to 1
        if importances.sum() > 0:
            importances = importances / importances.sum()
        
        return importances

---

## 3. Training & Optimization

### 3.1 Load and Prepare the Diabetes Dataset

In [ ]:
# Load the diabetes dataset
diabetes = load_diabetes()
X, y = diabetes.data, diabetes.target
feature_names = diabetes.feature_names

print("Dataset Information:")
print(f"  - Number of samples: {X.shape[0]}")
print(f"  - Number of features: {X.shape[1]}")
print(f"  - Feature names: {feature_names}")
print(f"  - Target range: [{y.min():.1f}, {y.max():.1f}]")
print(f"  - Target mean: {y.mean():.1f}")

In [ ]:
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Further split train into train and validation for early stopping analysis
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

print(f"Training set: {X_train_sub.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

### 3.2 Train the Model

In [ ]:
# Train with moderate settings for efficiency
gb_model = GradientBoostingRegressor(
    n_estimators=50,      # Number of trees
    learning_rate=0.1,    # Shrinkage parameter
    max_depth=3,          # Shallow trees work best
    min_samples_split=5,
    verbose=True
)

# Fit with validation tracking
gb_model.fit(X_train_sub, y_train_sub, X_val, y_val)

In [ ]:
# Evaluate on test set
y_pred = gb_model.predict(X_test)

print("\n" + "="*50)
print("Test Set Performance:")
print("="*50)
print(f"MSE:  {mean_squared_error(y_test, y_pred):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"R2:   {r2_score(y_test, y_pred):.4f}")

---

## 4. Diagnostics & Evaluation

### 4.1 MSE vs Number of Estimators

In [ ]:
# Plot training and validation loss curves
fig, ax = plt.subplots(figsize=(10, 6))

iterations = range(1, len(gb_model.train_losses) + 1)
ax.plot(iterations, gb_model.train_losses, 'b-', linewidth=2, label='Training MSE')
ax.plot(iterations, gb_model.val_losses, 'r-', linewidth=2, label='Validation MSE')

ax.set_xlabel('Number of Estimators', fontsize=12)
ax.set_ylabel('Mean Squared Error', fontsize=12)
ax.set_title('MSE vs Number of Estimators', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Mark the optimal point (minimum validation loss)
best_idx = np.argmin(gb_model.val_losses)
ax.axvline(x=best_idx + 1, color='g', linestyle='--', alpha=0.7, 
           label=f'Optimal: {best_idx + 1} estimators')
ax.scatter([best_idx + 1], [gb_model.val_losses[best_idx]], 
           color='g', s=100, zorder=5)

ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"Optimal number of estimators: {best_idx + 1}")
print(f"Minimum validation MSE: {gb_model.val_losses[best_idx]:.2f}")

### 4.2 Feature Importance

In [ ]:
# Calculate feature importances
importances = gb_model.feature_importances(X_test, y_test)

# Sort by importance
sorted_idx = np.argsort(importances)[::-1]

print("Feature Importances (Permutation-based):")
print("-" * 40)
for idx in sorted_idx:
    print(f"{feature_names[idx]:>10}: {importances[idx]:.4f}")

In [ ]:
# Visualize feature importances
fig, ax = plt.subplots(figsize=(10, 6))

y_pos = np.arange(len(feature_names))
sorted_importances = importances[sorted_idx]
sorted_names = [feature_names[i] for i in sorted_idx]

bars = ax.barh(y_pos, sorted_importances, color='steelblue', edgecolor='black')
ax.set_yticks(y_pos)
ax.set_yticklabels(sorted_names)
ax.invert_yaxis()  # Highest importance at top
ax.set_xlabel('Relative Importance', fontsize=12)
ax.set_title('Feature Importance (Permutation-based)', fontsize=14)

# Add value labels on bars
for bar, val in zip(bars, sorted_importances):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

### 4.3 Residual Analysis

In [ ]:
# Residual analysis
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Predicted vs Actual
axes[0].scatter(y_test, y_pred, alpha=0.6, edgecolors='black', linewidth=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', linewidth=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Values')
axes[0].set_ylabel('Predicted Values')
axes[0].set_title('Predicted vs Actual')
axes[0].legend()

# 2. Residuals vs Predicted
axes[1].scatter(y_pred, residuals, alpha=0.6, edgecolors='black', linewidth=0.5)
axes[1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Values')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residuals vs Predicted')

# 3. Residual distribution
axes[2].hist(residuals, bins=20, edgecolor='black', alpha=0.7)
axes[2].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[2].set_xlabel('Residuals')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

print(f"Residual statistics:")
print(f"  Mean: {residuals.mean():.2f}")
print(f"  Std:  {residuals.std():.2f}")
print(f"  Min:  {residuals.min():.2f}")
print(f"  Max:  {residuals.max():.2f}")

---

## 5. Visualizations

### 5.1 Training Progress: Staged Predictions

In [ ]:
# Track MSE at each stage
train_mse_stages = []
test_mse_stages = []

for y_train_pred in gb_model.staged_predict(X_train_sub):
    train_mse_stages.append(mean_squared_error(y_train_sub, y_train_pred))

for y_test_pred in gb_model.staged_predict(X_test):
    test_mse_stages.append(mean_squared_error(y_test, y_test_pred))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MSE vs iterations
iterations = range(1, len(train_mse_stages) + 1)
axes[0].plot(iterations, train_mse_stages, 'b-', linewidth=2, label='Training')
axes[0].plot(iterations, test_mse_stages, 'r-', linewidth=2, label='Test')
axes[0].set_xlabel('Number of Trees')
axes[0].set_ylabel('MSE')
axes[0].set_title('Training Progress: MSE vs Number of Trees')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Log scale view
axes[1].semilogy(iterations, train_mse_stages, 'b-', linewidth=2, label='Training')
axes[1].semilogy(iterations, test_mse_stages, 'r-', linewidth=2, label='Test')
axes[1].set_xlabel('Number of Trees')
axes[1].set_ylabel('MSE (log scale)')
axes[1].set_title('Training Progress (Log Scale)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 5.2 Effect of Learning Rate

In [ ]:
# Compare different learning rates
learning_rates = [0.01, 0.05, 0.1, 0.3, 0.5]
colors = plt.cm.viridis(np.linspace(0, 1, len(learning_rates)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

results = {}

for lr, color in zip(learning_rates, colors):
    # Train model with this learning rate
    model = GradientBoostingRegressor(
        n_estimators=50,
        learning_rate=lr,
        max_depth=3,
        verbose=False
    )
    model.fit(X_train_sub, y_train_sub, X_val, y_val)
    
    results[lr] = {
        'train_losses': model.train_losses,
        'val_losses': model.val_losses
    }
    
    # Plot training loss
    axes[0].plot(range(1, 51), model.train_losses, 
                 color=color, linewidth=2, label=f'lr={lr}')
    
    # Plot validation loss
    axes[1].plot(range(1, 51), model.val_losses,
                 color=color, linewidth=2, label=f'lr={lr}')

axes[0].set_xlabel('Number of Trees')
axes[0].set_ylabel('Training MSE')
axes[0].set_title('Training Loss for Different Learning Rates')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Number of Trees')
axes[1].set_ylabel('Validation MSE')
axes[1].set_title('Validation Loss for Different Learning Rates')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Summary table for learning rates
print("Learning Rate Comparison (at 50 trees):")
print("-" * 50)
print(f"{'Learning Rate':>15} {'Train MSE':>15} {'Val MSE':>15}")
print("-" * 50)
for lr in learning_rates:
    train_mse = results[lr]['train_losses'][-1]
    val_mse = results[lr]['val_losses'][-1]
    print(f"{lr:>15.2f} {train_mse:>15.2f} {val_mse:>15.2f}")

### 5.3 Effect of Max Depth

In [ ]:
# Compare different max depths
max_depths = [1, 2, 3, 4, 5]
colors = plt.cm.plasma(np.linspace(0, 0.8, len(max_depths)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

depth_results = {}

for depth, color in zip(max_depths, colors):
    model = GradientBoostingRegressor(
        n_estimators=50,
        learning_rate=0.1,
        max_depth=depth,
        verbose=False
    )
    model.fit(X_train_sub, y_train_sub, X_val, y_val)
    
    depth_results[depth] = {
        'train_losses': model.train_losses,
        'val_losses': model.val_losses
    }
    
    axes[0].plot(range(1, 51), model.train_losses,
                 color=color, linewidth=2, label=f'depth={depth}')
    axes[1].plot(range(1, 51), model.val_losses,
                 color=color, linewidth=2, label=f'depth={depth}')

axes[0].set_xlabel('Number of Trees')
axes[0].set_ylabel('Training MSE')
axes[0].set_title('Training Loss for Different Max Depths')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Number of Trees')
axes[1].set_ylabel('Validation MSE')
axes[1].set_title('Validation Loss for Different Max Depths')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary
print("\nMax Depth Comparison (at 50 trees):")
print("-" * 50)
print(f"{'Max Depth':>15} {'Train MSE':>15} {'Val MSE':>15}")
print("-" * 50)
for depth in max_depths:
    train_mse = depth_results[depth]['train_losses'][-1]
    val_mse = depth_results[depth]['val_losses'][-1]
    print(f"{depth:>15} {train_mse:>15.2f} {val_mse:>15.2f}")

---

## 6. Use Cases & Guidelines

### 6.1 When to Use Gradient Boosting

**Best scenarios:**

1. **Structured/Tabular Data**
   - Excel-like data with rows and columns
   - Mixed feature types (numerical + categorical)
   - Features have meaningful relationships

2. **Machine Learning Competitions**
   - Kaggle, competitions often won by gradient boosting
   - XGBoost, LightGBM, CatBoost are top choices
   - Excellent out-of-box performance

3. **When Accuracy is Priority**
   - State-of-the-art for many regression tasks
   - Often outperforms linear models and random forests

4. **Feature Interaction Detection**
   - Naturally captures non-linear relationships
   - Handles feature interactions without explicit engineering

5. **Moderate-sized Datasets**
   - Works well with thousands to millions of samples
   - Modern implementations (LightGBM) handle large data efficiently

### 6.2 When NOT to Use Gradient Boosting

**Avoid in these scenarios:**

1. **Very Limited Data (< 100 samples)**
   - Prone to overfitting with small datasets
   - Simpler models (linear regression) may generalize better

2. **Need for Interpretability**
   - Ensemble of trees is hard to explain
   - Regulatory requirements may demand interpretable models
   - Consider: Linear regression, decision trees, GAMs

3. **Real-time/Low-latency Predictions**
   - Sequential tree evaluation can be slow
   - Deep learning or linear models may be faster

4. **Unstructured Data**
   - Images, text, audio -> use deep learning
   - GBM doesn't capture spatial/sequential patterns well

5. **Extrapolation Required**
   - Trees cannot extrapolate beyond training data range
   - Linear models handle extrapolation better

6. **Streaming/Online Learning**
   - GBM is inherently batch learning
   - Difficult to update incrementally

### 6.3 Hyperparameter Tuning Tips

**Key hyperparameters and their effects:**

| Parameter | Typical Range | Effect | Tuning Strategy |
|-----------|--------------|--------|----------------|
| `n_estimators` | 100-1000 | More = better fit, longer training | Use early stopping |
| `learning_rate` | 0.01-0.3 | Lower = better generalization | Start with 0.1, lower if overfitting |
| `max_depth` | 3-8 | Higher = more complex trees | 3-5 usually optimal |
| `min_samples_split` | 2-20 | Higher = more regularization | Increase if overfitting |
| `subsample` | 0.5-1.0 | Row sampling, reduces overfitting | Try 0.8 |
| `colsample_bytree` | 0.5-1.0 | Feature sampling | Try 0.8 |

**Recommended tuning order:**
1. Set `n_estimators` high (1000) with early stopping
2. Tune `max_depth` and `min_samples_split`
3. Tune `subsample` and `colsample_bytree`
4. Lower `learning_rate` and increase `n_estimators`

### 6.4 Overfitting Prevention

**Strategies to prevent overfitting:**

1. **Early Stopping**
   - Monitor validation loss
   - Stop when validation loss stops improving

2. **Shrinkage (Learning Rate)**
   - Use smaller learning rate (0.01-0.1)
   - Requires more trees but generalizes better

3. **Tree Constraints**
   - Limit `max_depth` (3-5)
   - Increase `min_samples_split`
   - Limit `max_leaf_nodes`

4. **Subsampling**
   - Row subsampling (`subsample` < 1.0)
   - Column subsampling (`colsample_bytree` < 1.0)
   - Adds stochasticity like Random Forest

5. **Regularization** (in XGBoost/LightGBM)
   - L1 regularization (`reg_alpha`)
   - L2 regularization (`reg_lambda`)

In [ ]:
# Example: Demonstrating overfitting with high complexity
# This shows why we need regularization

# Overfit model (high depth, many trees, high learning rate)
overfit_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.5,
    max_depth=5,
    verbose=False
)
overfit_model.fit(X_train_sub, y_train_sub, X_val, y_val)

# Regularized model (shallow trees, low learning rate)
regularized_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=2,
    verbose=False
)
regularized_model.fit(X_train_sub, y_train_sub, X_val, y_val)

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(overfit_model.train_losses, 'b--', label='Overfit - Train', linewidth=2)
ax.plot(overfit_model.val_losses, 'b-', label='Overfit - Val', linewidth=2)
ax.plot(regularized_model.train_losses, 'g--', label='Regularized - Train', linewidth=2)
ax.plot(regularized_model.val_losses, 'g-', label='Regularized - Val', linewidth=2)

ax.set_xlabel('Number of Trees')
ax.set_ylabel('MSE')
ax.set_title('Overfitting vs Regularized Model')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Final Validation MSE:")
print(f"  Overfit model:     {overfit_model.val_losses[-1]:.2f}")
print(f"  Regularized model: {regularized_model.val_losses[-1]:.2f}")

---

## 7. Comparison with sklearn

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor as SklearnGBR
import time

### 7.1 Performance Comparison

In [ ]:
# Train our implementation
print("Training Custom Implementation...")
start_time = time.time()
custom_model = GradientBoostingRegressor(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=3,
    verbose=False
)
custom_model.fit(X_train, y_train)
custom_train_time = time.time() - start_time

custom_pred = custom_model.predict(X_test)
custom_mse = mean_squared_error(y_test, custom_pred)
custom_r2 = r2_score(y_test, custom_pred)

# Train sklearn implementation
print("Training sklearn Implementation...")
start_time = time.time()
sklearn_model = SklearnGBR(
    n_estimators=50,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
sklearn_model.fit(X_train, y_train)
sklearn_train_time = time.time() - start_time

sklearn_pred = sklearn_model.predict(X_test)
sklearn_mse = mean_squared_error(y_test, sklearn_pred)
sklearn_r2 = r2_score(y_test, sklearn_pred)

print("\nDone!")

In [ ]:
# Results comparison table
print("="*60)
print("Performance Comparison: Custom vs sklearn")
print("="*60)
print(f"{'Metric':<25} {'Custom':>15} {'sklearn':>15}")
print("-"*60)
print(f"{'MSE':<25} {custom_mse:>15.2f} {sklearn_mse:>15.2f}")
print(f"{'RMSE':<25} {np.sqrt(custom_mse):>15.2f} {np.sqrt(sklearn_mse):>15.2f}")
print(f"{'R2 Score':<25} {custom_r2:>15.4f} {sklearn_r2:>15.4f}")
print(f"{'Training Time (s)':<25} {custom_train_time:>15.4f} {sklearn_train_time:>15.4f}")
print("="*60)

### 7.2 Prediction Comparison

In [ ]:
# Visual comparison of predictions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Custom vs Actual
axes[0].scatter(y_test, custom_pred, alpha=0.6, edgecolors='black', linewidth=0.5)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', linewidth=2)
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_title(f'Custom Implementation\nR2={custom_r2:.4f}')

# sklearn vs Actual
axes[1].scatter(y_test, sklearn_pred, alpha=0.6, edgecolors='black', linewidth=0.5,
                color='orange')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 
             'r--', linewidth=2)
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].set_title(f'sklearn Implementation\nR2={sklearn_r2:.4f}')

# Custom vs sklearn
axes[2].scatter(custom_pred, sklearn_pred, alpha=0.6, edgecolors='black', linewidth=0.5,
                color='green')
axes[2].plot([custom_pred.min(), custom_pred.max()], 
             [custom_pred.min(), custom_pred.max()], 'r--', linewidth=2)
axes[2].set_xlabel('Custom Predictions')
axes[2].set_ylabel('sklearn Predictions')
correlation = np.corrcoef(custom_pred, sklearn_pred)[0, 1]
axes[2].set_title(f'Custom vs sklearn\nCorrelation={correlation:.4f}')

plt.tight_layout()
plt.show()

### 7.3 Feature Importance Comparison

In [ ]:
# Get feature importances from both
custom_importances = custom_model.feature_importances(X_test, y_test)
sklearn_importances = sklearn_model.feature_importances_

# Normalize sklearn importances
sklearn_importances = sklearn_importances / sklearn_importances.sum()

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(feature_names))
width = 0.35

bars1 = ax.bar(x - width/2, custom_importances, width, label='Custom (Permutation)', 
               color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, sklearn_importances, width, label='sklearn (Impurity)', 
               color='coral', edgecolor='black')

ax.set_xlabel('Features')
ax.set_ylabel('Relative Importance')
ax.set_title('Feature Importance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(feature_names, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### 7.4 Key Differences

| Aspect | Our Implementation | sklearn |
|--------|-------------------|--------|
| **Tree Building** | Greedy splits, limited thresholds | Optimized Cython implementation |
| **Feature Importance** | Permutation-based (computed on demand) | Impurity-based (during training) |
| **Loss Functions** | MSE only | MSE, MAE, Huber, Quantile |
| **Subsampling** | Not implemented | Row and column subsampling |
| **Speed** | Pure Python (slower) | Cython/C optimized (faster) |
| **Production Use** | Educational | Production-ready |

### 7.5 When to Use What

**Use our implementation when:**
- Learning how gradient boosting works
- Need to modify the algorithm for research
- Educational purposes

**Use sklearn when:**
- Production environments
- Need optimized performance
- Need additional features (different losses, subsampling)

**Consider XGBoost/LightGBM/CatBoost when:**
- Large datasets (millions of rows)
- Need GPU acceleration
- Kaggle competitions
- Need categorical feature handling

---

## Summary

In this notebook, we covered:

1. **Theory**: Boosting concept, gradient descent in function space, weak learners, learning rate, and loss functions

2. **Implementation**: Built a gradient boosting regressor from scratch using NumPy with decision trees as weak learners

3. **Training**: Applied to the diabetes dataset and tracked training/validation progress

4. **Diagnostics**: Analyzed MSE curves, feature importance, and residuals

5. **Visualizations**: Explored effects of learning rate and max depth on model performance

6. **Guidelines**: Discussed when to use (and not use) gradient boosting, hyperparameter tuning, and overfitting prevention

7. **Comparison**: Validated our implementation against sklearn's GradientBoostingRegressor

**Key Takeaways:**
- Gradient boosting builds models sequentially, each correcting previous errors
- Lower learning rates with more trees generally perform better
- Shallow trees (depth 3-5) work best as weak learners
- Use early stopping to prevent overfitting
- For production, use sklearn, XGBoost, LightGBM, or CatBoost